# Extraction Agent — local models (Qwen3-VL-8B + Molmo2 + PaddleOCR), GPU-only (Colab)

Local-model counterpart to the recorded GPT-5.5/Sonnet/Gemini extraction-agent baselines
(`history/model_comparison_1782539273.json`). Runs the REAL, UNMODIFIED
`pnid_pipeline.extract.extract_page` on the 13 real sheets this project already has
`reviewed_truth.json` ground truth for, with local models injected at the pipeline's own
extension points:

```
PaddleOCR (CPU)  ->  real extract_page() (either real pipeline mode)
  -> Qwen3-VL-8B (GPU, injected call_llm)  -> optional Molmo2 (GPU) supplementary slot
  -> score vs reviewed_truth.json (real score.py functions)
```

**This CANNOT run locally** — no usable GPU on the dev machine for an 8B VLM + a 7B
pointing model. Colab-only, same established pattern as `ArmL_QwenVL_FullStack_GPUOnly.ipynb`
/ `ArmL_Molmo2_Qwen_Mixed_GPUOnly.ipynb`. No Google Drive anywhere — HF is the only shared
storage channel (MEMORY.md: "No Google Drive, ever").

## Deliberate exception to the CPU/GPU split (H6, same precedent as ArmL)

This project's standing rule is "new notebooks split CPU-prep (local + HF) from GPU-only
Colab cells." This notebook follows that split MORE than ArmL did, actually — Phase A
(Molmo2 + PaddleOCR + rendering) and Phase B (Qwen3-VL + `extract_page`) are two genuinely
separate model-loading phases *within* this one Colab session, by design (§6 of
`Extraction_Agent_Local_Plan.md`: "Two-phase-per-corpus model schedule ... avoids
co-residency"), so Molmo2 and Qwen3-VL are never resident on the GPU at the same time.
The H6 exception that DOES still apply here: both phases still have to run in the SAME
Colab process/session, because Phase B's `L-ocr+M`/`L-cv+M` configs consume Phase A's cached
Molmo2 points directly (no separate CPU-only notebook is possible for either phase — Phase A
needs a GPU for Molmo2, Phase B needs a GPU for Qwen3-VL). Per-sheet caching to disk between
phases (`molmo_points/<stem>.json`, `ocr_words/<stem>.json`) is what makes the handoff safe
memory-wise, matching the plan's exact schedule.

## Honesty status (read before trusting any output this notebook produces)

**Nothing in this notebook has been run end-to-end.** No GPU is available in the environment
that wrote it. Every real-model call path (`qwen_generate.py`, `molmo_points.py`) was built
by extracting the PROVEN load/generate recipe from this project's own working notebooks
(cited inline below), but:
- Whether Qwen3-VL-8B reliably produces valid `_OneTag`/`_ManyTags`/`OcrResult` JSON under
  THIS pipeline's own real prompts (richer/different schemas than any prior Qwen benchmark
  in this project) is **UNTESTED**. Per-call parse-failure tracking + the empty-tag/salvage
  fallback (`qwen_call_llm.py`) are the mitigation, not a substitute for actually running it.
- Molmo2's transfer to these specific dense industrial sheets, at the 512px/2x config
  validated on Gupta-style symbol sheets, is **UNTESTED** (CLAUDE.md/plan §9 risk 2) — a
  net-negative `+M` result is a valid, reportable finding, not a bug.
- `L-cv+M`'s Molmo `snap_candidates` merge **IS wired as of 2026-07-17** (plan §11, Phase 1
  gap item 3 closed): `_install_cv_molmo_snap_wrapper` in `run_extraction_local.py`
  monkeypatches `pnid_pipeline.extract.snap_candidates` (one-shot per run, in-place `symbols`
  extension) — zero agent-source edits, CPU fake-backed end-to-end verified (a
  `molmo_point`-signaled tag reached the final preds JSON). Like everything else here, it has
  **never been GPU-tested** with real Molmo2 points.


## ⚠️ USER CHECKPOINT — RESOLVED 2026-07-17

The 13 real sheet PDFs (`AG_PNID`/`RIVE_LTTS_Sample` trees) are marked **Restricted/EAR99**.
They are **NOT** included in `package_extraction_agent_src_for_colab.sh`'s zip (see that
script's header). Tom gave explicit, separate sign-off for this specific upload
(Extraction_Agent_Local_Plan.md §10 checkpoint 1, resolved) and both archives are now
pushed to the private `DATA_REPO` (`timthy45/pnid-extraction-datasets`) at
`sheets/AG_PNID.zip` / `sheets/RIVE_LTTS_Sample.zip` — confirmed present and confirmed
`private=True` via the HF API before §4 below was filled in.

**Note on internal zip structure:** the two archives are single-level
(`AG_PNID.zip` → `AG_PNID/<file>.pdf`, `RIVE_LTTS_Sample.zip` → `RIVE/<file>.pdf`), NOT the
double-nested `AG_PNID/AG_PNID/...` shape the local scratchpad folders have — §1's `AG_DIR`/
`RIVE_DIR` were set to match the real uploaded structure, verified via `unzip -l` before
being written, not assumed.


In [1]:
!nvidia-smi

Mon Jul 20 12:11:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Config

In [1]:
# ── Model config ─────────────────────────────────────────────────────────
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MOLMO_MODEL_ID = "allenai/Molmo2-O-7B"
QWEN_MAX_NEW_TOKENS = 4096          # capped further per-call inside run_extraction_local
                                     # via Phase 1c fix 3(a)'s dense_chunk_tokens=800 config edit

# ── HF (datasets, private agent-code zip). No Google Drive, ever. ──────────
import os
# Token is read from the environment, never hardcoded. In Colab add it under
# Secrets (key: HF_TOKEN) and enable notebook access; locally just export it.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN is not set - add it to Colab Secrets or export it") from e
DATA_REPO = "timthy45/pnid-extraction-datasets"                 # results push target (established repo)
EXTRACTION_AGENT_SRC_REPO = "timthy45/pnid-extraction-agent-src"  # PRIVATE — created by
                                                                    # scripts/package_extraction_agent_src_for_colab.sh
                                                                    # (run LOCALLY first, see bottom of this notebook)
EXTRACTION_AGENT_SRC_FILE = "agent_src/latest.zip"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "PASTE_YOUR_HF_TOKEN_HERE", "paste your HF token"

# ── Reference baselines (hardcoded from the REAL recorded numbers — NOT recomputed here) ──
# Extraction_Agent_Local_Plan.md §2, sourced verbatim from
# agents/pnid-extraction-agent/scripts/eval/history/model_comparison_1782539273.json
GPT55_HIGH_MEAN_REVR = 0.836   # OPENAI_REASONING_EFFORT=high (proxy default), 14 sheets
GPT55_LOW_MEAN_REVR = 0.813    # OPENAI_REASONING_EFFORT=low, 14 sheets
SONNET46_MEAN_REVR = 0.811
GEMINI31PRO_MEAN_REVR = 0.752
# NOTE: the recorded baselines cover 14 sheets (includes Sample_PID); this notebook's 13
# sheets exclude Sample_PID (no PDF in our packages, per Extraction_Agent_Local_Plan.md §1)
# — the 3-way comparison in §8 below reports this discrepancy explicitly, does not pretend
# it is the same 14.

# ── The 13 sheets. Paths match the ACTUAL internal structure of the zips pushed to HF
# (verified 2026-07-17 via `unzip -l` before this cell was filled in — the zips are
# single-level: AG_PNID.zip -> "AG_PNID/<file>.pdf", RIVE_LTTS_Sample.zip -> "RIVE/<file>.pdf",
# NOT the double-nested "AG_PNID/AG_PNID/..." the local scratchpad folder structure has).
# Stems/filenames themselves reused verbatim from src/e2e_harness/score_revR_real_sheets.py's
# SHEETS list — only the directory prefix differs from that file, do not re-derive the rest. ──
AG_DIR = "/content/sheets/AG_PNID"
RIVE_DIR = "/content/sheets/RIVE"

SHEETS = [
    ("GD-B-540-DP-2920-005", f"{AG_DIR}/GD-B-540-DP-2920-005-Z.pdf"),
    ("GD-B-550-DP-3322-003", f"{AG_DIR}/GD-B-550-DP-3322-003-Z2.pdf"),
    ("GD-B-615-DP-1148-006", f"{AG_DIR}/GD-B-615-DP-1148-006-Z2.pdf"),
    ("GD-H-375-DP-2590-003", f"{AG_DIR}/GD-H-375-DP-2590-003-Zpdf.pdf"),
    ("GD-T-435-DR-2031-030", f"{AG_DIR}/GD-T-435-DR-2031-030-Z2.pdf"),
    ("GD-T-435-DT-2042-056", f"{AG_DIR}/GD-T-435-DT-2042-056-Z.pdf"),
    ("PX-2365-0140006-001", f"{RIVE_DIR}/PX-2365-0140006-001.PDF"),
    ("PX-2365-0140031-001", f"{RIVE_DIR}/PX-2365-0140031-001.PDF"),
    ("PX-2365-0150022-001", f"{RIVE_DIR}/PX-2365-0150022-001.pdf"),
    ("PX-2365-0150033-008", f"{RIVE_DIR}/PX-2365-0150033-008.pdf"),
    ("PX-2365-9850077-001", f"{RIVE_DIR}/PX-2365-9850077-001.pdf"),
    ("PX-2368-0180004-001", f"{RIVE_DIR}/PX-2368-0180004-001.pdf"),
    ("PX-2368-0180021-002", f"{RIVE_DIR}/PX-2368-0180021-002.pdf"),
]

# §7 dev-tuning subset (ONLY these get tunable-parameter iteration, per the plan's honesty
# rule about dev/held-out split reporting)
DEV_STEMS = {"PX-2368-0180004-001", "GD-T-435-DR-2031-030", "PX-2365-0150022-001"}
SMOKE_STEM = "PX-2368-0180004-001"   # n=50, human-curated, GPT-5.5 scored 0.98 revR here


In [2]:
import os, shutil
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms_standalone.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/arms_standalone.py")
get_ipython().system_raw(f"cd /content && HF_TOKEN={HF_TOKEN} nohup python3 arms_standalone.py > /content/arms.log 2>&1 &")
print("launched in background")

arms_standalone.py:   0%|          | 0.00/20.0k [00:00<?, ?B/s]

launched in background


In [3]:
import os, shutil
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["ARMS_SHEETS"] = "GD-B-540-DP-2920-005,GD-B-615-DP-1148-006,PX-2365-0140006-001,PX-2365-9850077-001"
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms_standalone.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
shutil.copy(_p, "/content/arms_standalone.py")
get_ipython().system_raw(f"cd /content && HF_TOKEN={HF_TOKEN} ARMS_SHEETS={os.environ['ARMS_SHEETS']} nohup python3 arms_standalone.py > /content/arms.log 2>&1 &")
print("launched on 4 new sheets:", os.environ["ARMS_SHEETS"])

arms_standalone.py:   0%|          | 0.00/20.0k [00:00<?, ?B/s]

launched on 4 new sheets: GD-B-540-DP-2920-005,GD-B-615-DP-1148-006,PX-2365-0140006-001,PX-2365-9850077-001


In [14]:
!tail -50 /content/arms.log

[12:12:03] installing/verifying deps (no-ops when already satisfied)...
[12:12:44] downloading private agent code...
[12:12:50] downloading sheet PDFs...
[12:12:57] sheets this run: ['GD-B-540-DP-2920-005', 'GD-B-615-DP-1148-006', 'PX-2365-0140006-001', 'PX-2365-9850077-001']
[12:12:58] ARM 0 (deterministic, no model)
[12:12:59]   GD-B-540-DP-2920-005 arm0 revR=0.028 (3/109)
[12:12:59]   GD-B-615-DP-1148-006 arm0 revR=0.086 (19/220)
[12:12:59]   PX-2365-0140006-001 arm0 revR=0.149 (37/248)
[12:13:00]   PX-2365-9850077-001 arm0 revR=0.000 (0/131)
[12:13:04] loading Qwen3-VL-8B...
2026-07-20 12:13:08.825613: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 12:13:08.897727: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to 

In [16]:
!tail -20 /content/arms.log
!nvidia-smi

- video_processing_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/allenai/Molmo2-O-7B:
- image_processing_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/allenai/Molmo2-O-7B:
- processing_molmo2.py
- video_processing_molmo2.py
- image_processing_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. Thi

In [2]:
!pkill -f arms_standalone.py

In [3]:
!ps aux | grep arms_standalone

root        3606  0.0  0.0   7376  3588 ?        S    11:35   0:00 /bin/bash -c ps aux | grep arms_standalone
root        3608  0.0  0.0   6616  2352 ?        S    11:35   0:00 grep arms_standalone


In [ ]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/install_cell.py", repo_type="dataset", token=HF_TOKEN)
exec(open(_p).read())

+ pip install transformers==4.57.1 accelerate huggingface_hub
+ pip install pymupdf python-magic pydantic
+ pip install numpy==2.3.5
+ pip install pillow<12
[preflight] fresh-interpreter check: OK
            To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Pillow 11.3.0 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | transformers 4.57.1
CUDA available: True
NVIDIA A100-SXM4-80GB 85 GB

S2 environment OK - continue with S3 onward.


## 2. Install (GPU-runtime prep)

`transformers==4.57.1` pinned — same version this project's other Qwen3-VL/Molmo2 notebooks
use (known-good for `Qwen3-VL-8B-Instruct` + `AutoModelForImageTextToText`, and for Molmo2's
`trust_remote_code` processor). PaddleOCR is CPU-only — installed here but never competes
with Qwen/Molmo2 for VRAM.


In [4]:
!apt-get -qq update && apt-get -qq install -y libmagic1 > /dev/null
!pip install -q transformers==4.57.1 accelerate huggingface_hub
!pip install -q paddleocr paddlepaddle
!pip install -q pymupdf python-magic pydantic

# Fix history: (1) Pillow 12.0.0 broke PIL._typing._Ink (known upstream regression);
# (2) torchvision was uninstalled by an earlier wrong fix and pip changes survive
# kernel restarts; (3) reinstalling torchvision from plain PyPI gave an ABI-mismatched
# wheel ("operator torchvision::nms does not exist") because Colab's torch is a
# CUDA-specific build. Fix: install torchvision from the PyTorch index matching
# torch's own CUDA tag, then pin pillow<12 last.
import torch as _torch_pre
_ver, _, _local = _torch_pre.__version__.partition("+")   # e.g. ("2.8.0", "+", "cu126")
if _local.startswith("cu"):
    _idx = f"https://download.pytorch.org/whl/{_local}"
    !pip install -q torchvision "torch=={_ver}" --index-url {_idx}
else:
    !pip install -q torchvision "torch=={_ver}"

!pip install -q "pillow<12"

# Fail-fast verification — the exact import chains that crashed the real runs:
import importlib.metadata as _md
import PIL
from PIL import ImageDraw
import torchvision
from transformers import AutoModelForImageTextToText, AutoProcessor
import paddleocr

print("Pillow", PIL.__version__, "| torchvision", torchvision.__version__,
      "| transformers", _md.version("transformers"), "| paddleocr", _md.version("paddleocr"))
assert not PIL.__version__.startswith("12."), "Pillow 12.x still present -- pin failed"
_disk_torch = _md.version("torch").split("+")[0]
if not _torch_pre.__version__.startswith(_disk_torch):
    raise RuntimeError(f"torch changed on disk ({_torch_pre.__version__} loaded vs "
                       f"{_disk_torch} installed) - Runtime -> Restart session, then rerun this cell once")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

SyntaxError: invalid syntax (371879387.py, line 32)

## 3. Private code: pnid-extraction-agent + e2e_bench + extraction_local

Downloads the zip `scripts/package_extraction_agent_src_for_colab.sh` builds and pushes
(run that LOCALLY first — see the run-order note at the bottom of this notebook). This zip
does **NOT** contain the 13 sheet PDFs (see the USER CHECKPOINT cell above) — only code +
the `reviewed_truth.json` ground truth + `score.py`.

`pnid_pipeline` is NOT pip-installed (no `pyproject.toml`/`setup.py` by design, confirmed in
`Extraction_Agent_Local_Plan.md` §11's "Environment setup" note) — made importable the same
way `.venv-e2e` does it on the Mac: a `.pth`-style `sys.path` insert pointing at the agent's
repo root, done here via a plain `sys.path.insert` (Colab has no venv site-packages dir to
drop a `.pth` file into conveniently).


In [3]:
import zipfile, sys
from pathlib import Path
from huggingface_hub import hf_hub_download

AGENT_SRC_ROOT = Path("/content/agent_src")
AGENT_SRC_ROOT.mkdir(exist_ok=True)

zp = hf_hub_download(repo_id=EXTRACTION_AGENT_SRC_REPO, filename=EXTRACTION_AGENT_SRC_FILE,
                      repo_type="dataset", token=HF_TOKEN)
with zipfile.ZipFile(zp) as zf:
    zf.extractall(AGENT_SRC_ROOT)
print("extracted:", sorted(p.name for p in AGENT_SRC_ROOT.iterdir()))

AGENT_DIR = str(AGENT_SRC_ROOT / "agents" / "pnid-extraction-agent")
PID_ML_SRC = str(AGENT_SRC_ROOT / "pid_ml_src")
for p in (AGENT_DIR, PID_ML_SRC):
    if p not in sys.path:
        sys.path.insert(0, p)

!pip install -q pdfplumber   # path_a.py (Path A text-layer extraction) needs this,
                              # per Extraction_Agent_Local_Plan.md ''s Environment setup note

import importlib
REQUIRED_MODULES = [
    "pnid_pipeline.extract", "pnid_pipeline.vision", "pnid_pipeline.ocr_reasoning",
    "pnid_pipeline.grounded_read", "pnid_pipeline.triage", "pnid_pipeline.rasterize",
    "pnid_pipeline.run",
    "e2e_bench.backends.parse_json_common", "e2e_bench.backends.parse_molmo", "e2e_bench.types",
    "extraction_local.qwen_call_llm", "extraction_local.paddle_ocr",
    "extraction_local.molmo_candidates", "extraction_local.molmo_synthetic_tokens",
    "extraction_local.molmo_render", "extraction_local.run_extraction_local",
]
missing = []
for mod in REQUIRED_MODULES:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f"{mod}: {type(e).__name__}: {e}")
if missing:
    raise RuntimeError("Missing/broken imports:\n  " + "\n  ".join(missing))
print(f"all {len(REQUIRED_MODULES)} required modules import cleanly")

# scripts/eval is a directory of loose scripts, not a package with __init__ guarantees for
# every submodule — import it exactly the way run_extraction_local.py already does locally.
sys.path.insert(0, AGENT_DIR)
from scripts.eval.score import load_reviewed_truth, review_keep, review_recall
print("scripts.eval.score imported (real revR scorer)")


agent_src/latest.zip:   0%|          | 0.00/620k [00:00<?, ?B/s]

extracted: ['agents', 'pid_ml_src']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 154.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 155.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 130.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
all 16 required modules import cleanly
scripts.eval.score imported (real revR scorer)


## 4. Sheet PDFs — download (checkpoint resolved, see cell above)

Both archives are on HF now. Extracts to `/content/sheets/AG_PNID/*` and
`/content/sheets/RIVE/*` — matching the real single-level zip structure (see §1's `AG_DIR`/
`RIVE_DIR`), not the double-nested local scratchpad layout.


In [4]:
# RESOLVED 2026-07-17: Tom gave explicit sign-off (Extraction_Agent_Local_Plan.md §10
# checkpoint 1) and both sheet archives are pushed to HF at DATA_REPO's sheets/ path
# (verified present + DATA_REPO confirmed private=True via the HF API before this cell
# was filled in — not just trusted an upload script's exit message).

from huggingface_hub import hf_hub_download
import zipfile

for fname in ("AG_PNID.zip", "RIVE_LTTS_Sample.zip"):
    zp = hf_hub_download(repo_id=DATA_REPO, filename=f"sheets/{fname}",
                          repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall("/content/sheets")
    print(f"extracted {fname}")

from pathlib import Path
missing_sheets = [(stem, p) for stem, p in SHEETS if not Path(p).exists()]
if missing_sheets:
    print(f"{len(missing_sheets)}/{len(SHEETS)} sheet PDFs not present yet (expected until "
          f"the checkpoint above is resolved):")
    for stem, p in missing_sheets:
        print(f"  MISSING  {stem}: {p}")
else:
    print(f"all {len(SHEETS)} sheet PDFs present.")


sheets/AG_PNID.zip:   0%|          | 0.00/63.0M [00:00<?, ?B/s]

extracted AG_PNID.zip


sheets/RIVE_LTTS_Sample.zip:   0%|          | 0.00/20.3M [00:00<?, ?B/s]

extracted RIVE_LTTS_Sample.zip
all 13 sheet PDFs present.


## 5. Phase A — Molmo2: render + PaddleOCR + per-class pointing, ALL 13 sheets, then free

Per §6's two-phase schedule: Molmo2 loads, runs over every sheet, caches
`molmo_points/<stem>.json` (+ `ocr_words/<stem>.json`, real OCR words at the SAME
`molmo_render_page` coordinate scale, needed later for `L-ocr+M`'s near/pair-radius
pairing) to disk, then is FREED before Qwen3-VL ever loads (memory-safe on a 24GB GPU,
per the plan — avoids co-residency).

Uses `molmo_render.molmo_render_page` (Phase 1c fix 1: real `triage_page`->`work_zoom`->
`render_page` sequence, rotation-correct) — NOT a raw `pymupdf` render at some arbitrary
DPI — so Molmo2's points land in EXACTLY the coordinate space `extract_page` itself uses
internally. This is the single most important correctness property of this whole notebook;
see `molmo_render.py`'s docstring for the full rationale.


In [5]:
from extraction_local.molmo_points import load_molmo_model, molmo_point_classes, DEFAULT_CLASSES
from extraction_local.molmo_render import molmo_render_page
from extraction_local.paddle_ocr import paddle_ocr_words
from pnid_pipeline.run import load_config, _load_env
import numpy as np, json, os, time

os.makedirs("/content/molmo_points", exist_ok=True)
os.makedirs("/content/ocr_words", exist_ok=True)

_load_env()
_cfg = load_config()

molmo_model, molmo_processor = load_molmo_model(MOLMO_MODEL_ID)
print("Molmo2-O-7B loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Molmo2 will point for these classes:", DEFAULT_CLASSES)

for stem, pdf_path in SHEETS:
    if not Path(pdf_path).exists():
        print(f"SKIP {stem}: PDF not present (see §4 checkpoint)")
        continue
    t0 = time.time()
    img, W, H, zoom = molmo_render_page(pdf_path, 0, _cfg)   # rotation-correct render (Phase 1c fix 1)

    # PaddleOCR on this SAME rendered array — cached alongside Molmo points so Phase B''s
    # L-ocr+M wrapper (molmo_synthetic_tokens pairing) does not need to re-render/re-OCR.
    import cv2
    bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    ocr_words = paddle_ocr_words(bgr)

    points_by_class = molmo_point_classes(molmo_model, molmo_processor, img, DEFAULT_CLASSES)

    with open(f"/content/molmo_points/{stem}.json", "w") as f:
        json.dump({"W": W, "H": H, "zoom": zoom, "points_by_class": points_by_class}, f)
    with open(f"/content/ocr_words/{stem}.json", "w") as f:
        json.dump(ocr_words, f)

    n_points = sum(len(v) for v in points_by_class.values())
    print(f"{stem:25s} W={W} H={H} zoom={zoom:.2f}  molmo_points={n_points:4d}  "
          f"ocr_words={len(ocr_words):4d}  ({time.time()-t0:.1f}s)")

del molmo_model, molmo_processor
torch.cuda.empty_cache()
print("Molmo2 freed. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


image_processing_molmo2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/allenai/Molmo2-O-7B:
- image_processing_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/allenai/Molmo2-O-7B:
- video_processing_molmo2.py
- image_processing_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


preprocessor_config.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Molmo2-O-7B loaded. VRAM: 31.0 GB
Molmo2 will point for these classes: ['valve', 'instrument bubble', 'pump', 'vessel or tank', 'off-page connector', 'equipment']


: 

: 

: 

## 6. Phase B — Qwen3-VL-8B: load, then run all 4 configs per sheet

`build_qwen_generate_fn`/`load_qwen_model` — verbatim recipe from
`ArmL_QwenVL_FullStack_GPUOnly.ipynb` (see `qwen_generate.py`'s module docstring for the
exact cell/notebook citation). Base model, zero-shot, no LoRA adapter — no adapter in this
project has ever been validated for extraction-agent reads specifically, so base is the
honest starting point (same reasoning `ArmL_QwenVL_FullStack_GPUOnly.ipynb` documents for
its own detection call).


In [5]:
from extraction_local.qwen_generate import load_qwen_model, build_qwen_generate_fn

qwen_model, qwen_processor = load_qwen_model(QWEN_MODEL_ID)
print("Qwen3-VL-8B (base) loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

qwen_generate_fn = build_qwen_generate_fn(qwen_model, qwen_processor,
                                           max_new_tokens_default=QWEN_MAX_NEW_TOKENS)
print("qwen_generate_fn ready")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL-8B (base) loaded. VRAM: 17.5 GB
qwen_generate_fn ready


## 7. Run all 4 configs over all 13 sheets

Per §6/§7 of the plan:
- **`L-ocr`** — real default `ocr_reasoning` mode, NO Molmo. TRUE apples-to-apples vs. the
  recorded GPT-5.5 baselines.
- **`L-ocr+M`** — same, + Molmo2 synthetic-token injection (cached points from Phase A).
- **`L-cv`** — `PNID_MODE=cv` (CV-hybrid `read_shapes`/`read_regions`), NO Molmo. Bonus
  second architecture, not directly comparable to the recorded baselines (see plan §11
  discovery: the real default is `ocr_reasoning`, not `cv`).
- **`L-cv+M`** — same `PNID_MODE=cv` config, + Molmo2 candidates merged into the CV path's
  real `snap_candidates` merge (**wired 2026-07-17**, closing Phase 1 build-log gap item 3:
  `_install_cv_molmo_snap_wrapper` monkeypatches `pnid_pipeline.extract.snap_candidates`,
  injects `molmo_candidates(...)` on the first snap call of each run and extends `symbols`
  in place so adjudication sees the Molmo boxes — zero agent-source edits, fake-backed
  end-to-end verified on CPU, never GPU-tested).

Each cell run is wall-clocked and prints its own `call_llm.usage` (real generation-call
count — the plan's honesty requirement for reporting local generation counts, since
concurrency collapses to 1 for local serial GPU generation, unlike the recorded API-based
baselines).


In [6]:
from extraction_local.run_extraction_local import run_one_sheet, score_against_reviewed_truth

def run_config(pipeline_mode: str, use_molmo: bool, config_name: str):
    rows = []
    for stem, pdf_path in SHEETS:
        if not Path(pdf_path).exists():
            print(f"SKIP {stem}: PDF not present (see §4 checkpoint)")
            continue
        molmo_cache_path = f"/content/molmo_points/{stem}.json"
        molmo_points_by_class = None
        if use_molmo:
            if not Path(molmo_cache_path).exists():
                print(f"  [warn] {stem}: no cached Molmo points (Phase A skipped this sheet) "
                      f"-> running WITHOUT Molmo for this sheet")
            else:
                with open(molmo_cache_path) as f:
                    molmo_points_by_class = json.load(f)["points_by_class"]

        t0 = time.time()
        try:
            out = run_one_sheet(pdf_path, stem, pipeline_mode=pipeline_mode,
                                 generate_fn=qwen_generate_fn,
                                 molmo_points_by_class=molmo_points_by_class,
                                 preds_dir=f"/content/preds_{config_name}")
        except Exception as e:
            print(f"  [FAIL] {stem} ({config_name}): {type(e).__name__}: {e}")
            continue
        elapsed = time.time() - t0

        scored = score_against_reviewed_truth(stem, out["dumped"])
        usage = out["call_llm"].usage
        n_calls = sum(v.get("calls", 0) for v in usage.values())
        row = {"stem": stem, "config": config_name, "revR": scored["revR"],
               "hits": scored["hits"], "n_truth": scored["n_truth"],
               "n_pred_clean": scored["n_pred_clean"], "sec_per_sheet": round(elapsed, 1),
               "n_generations": n_calls, "dev_sheet": stem in DEV_STEMS}
        rows.append(row)
        print(f"  {stem:25s} {config_name:10s} revR={row['revR']:.3f} "
              f"({row['hits']}/{row['n_truth']})  {elapsed:.0f}s  {n_calls} generations")
    return rows

all_rows = []


### 7.1 Smoke gate (1 sheet, before spending on the full 13)

Per §7 run protocol: parse-failure rate < 20%, some candidates from every source, wall-clock
≤ ~25 min/sheet. **STOP and report to Tom if any gate fails** — do not silently continue to
the full run (§10 checkpoint 2).


In [7]:
import time, json, os
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [15]:
SMOKE_ONLY = True   # flip to False only after the smoke gates below pass and Tom says go

if SMOKE_ONLY:
    smoke_pdf = dict(SHEETS)[SMOKE_STEM]
    print(f"=== SMOKE: {SMOKE_STEM} (L-ocr) ===")
    t0 = time.time()
    out = run_one_sheet(smoke_pdf, SMOKE_STEM, pipeline_mode="ocr_reasoning",
                         generate_fn=qwen_generate_fn, preds_dir="/content/preds_smoke")
    elapsed = time.time() - t0
    scored = score_against_reviewed_truth(SMOKE_STEM, out["dumped"])
    n_calls = sum(v.get("calls", 0) for v in out["call_llm"].usage.values())
    n_tags = len(out["dumped"].get("tags", []))
    print(f"revR={scored['revR']:.3f}  tags={n_tags}  generations={n_calls}  {elapsed:.0f}s")
    print(f"Gate: wall-clock <= ~1500s/sheet: {'PASS' if elapsed <= 1500 else 'FAIL'}")
    print(f"Gate: at least one tag produced: {'PASS' if n_tags > 0 else 'FAIL'}")
    print("Gate: parse-failure rate < 20% -- inspect printed [parse-fail] lines above, if any "
          "(build_qwen_call_llm has no separate failure counter exposed yet; per-call parse "
          "failures fall back to an empty/salvaged tag silently by design -- eyeball the raw "
          "revR/tag-count numbers above against expectations for this sheet, GPT-5.5 scored "
          "0.98 revR here).")
    print("\nSTOP: get Tom''s go/no-go on these gates before setting SMOKE_ONLY=False.")


=== SMOKE: PX-2368-0180004-001 (L-ocr) ===


: 

: 

: 

In [1]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/cached_ocr_run3.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

NameError: name 'HF_TOKEN' is not defined

### 7.2 Full run — all 4 configs, all 13 sheets (only after the smoke gate passes)

In [12]:
if not SMOKE_ONLY:
    print("=== L-ocr (real default mode, no Molmo) ===")
    all_rows += run_config("ocr_reasoning", use_molmo=False, config_name="L-ocr")

    print("\n=== L-ocr+M (real default mode + Molmo synthetic tokens) ===")
    all_rows += run_config("ocr_reasoning", use_molmo=True, config_name="L-ocr+M")

    print("\n=== L-cv (PNID_MODE=cv, no Molmo) ===")
    all_rows += run_config("cv", use_molmo=False, config_name="L-cv")

    print("\n=== L-cv+M (PNID_MODE=cv + Molmo via the snap_candidates merge) ===")
    # REAL ablation as of 2026-07-17 (plan SS11): run_one_sheet(pipeline_mode="cv",
    # molmo_points_by_class=...) installs _install_cv_molmo_snap_wrapper, which merges
    # molmo_candidates(...) into the CV path's first snap_candidates call and extends
    # `symbols` in place. Cached Phase A points are in the correct pixel space by
    # construction (molmo_render_page replicates _path_b_candidates' exact render).
    all_rows += run_config("cv", use_molmo=True, config_name="L-cv+M")
else:
    print("SMOKE_ONLY is True -- full run skipped. Set SMOKE_ONLY=False after the smoke gate "
          "passes and Tom gives the go-ahead (§10 checkpoint 2).")


NameError: name 'SMOKE_ONLY' is not defined

In [14]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/cached_ocr_run3.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

cached_ocr_run3.py: 0.00B [00:00, ?B/s]

precomputed_ocr_words.json: 0.00B [00:00, ?B/s]

cached OCR sheets available: {'PX-2368-0180004-001': 564, 'GD-T-435-DR-2031-030': 358, 'PX-2365-0150022-001': 674}
cached-OCR shim installed (stem-keyed) - PaddleOCR will not run in Colab
running sheets: ['GD-T-435-DR-2031-030', 'PX-2365-0150022-001', 'PX-2368-0180004-001']


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  GD-T-435-DR-2031-030      L-ocr revR=0.000 (0/0)  7s  2 generations   [GPT-5.5-high recorded: 0.841]
  PX-2365-0150022-001       L-ocr revR=0.000 (0/0)  6s  2 generations   [GPT-5.5-high recorded: 0.75]
  PX-2368-0180004-001       L-ocr revR=0.000 (0/0)  6s  2 generations   [GPT-5.5-high recorded: 0.98]

L-ocr done - run the S8 scoring cell, or read the per-sheet lines above directly.


## 8. Scoring + reporting (§8 of the plan)

Headline 3-way comparison, per-sheet table, dev-vs-held-out split (dev = the 3 sheets in
`DEV_STEMS`), AG-vs-PX family split. **Do not write `results.csv` or any report until Tom has
seen these raw numbers (§10 checkpoint 3).**


In [ ]:
import statistics

def summarize(rows, label_filter=None):
    r = [x for x in rows if label_filter is None or label_filter(x)]
    if not r:
        return None
    return {"n": len(r), "mean_revR": round(statistics.mean(x["revR"] for x in r), 3)}

if all_rows:
    configs = sorted(set(r["config"] for r in all_rows))
    print("=== Headline: recorded GPT-5.5 baseline vs local configs ===")
    print(f"GPT-5.5 high (recorded, 14 sheets): {GPT55_HIGH_MEAN_REVR}")
    print(f"GPT-5.5 low  (recorded, 14 sheets): {GPT55_LOW_MEAN_REVR}")
    print(f"Sonnet 4.6   (recorded, 14 sheets): {SONNET46_MEAN_REVR}")
    print(f"Gemini 3.1 Pro (recorded, 14 sheets): {GEMINI31PRO_MEAN_REVR}")
    for c in configs:
        s_all = summarize(all_rows, lambda x: x["config"] == c)
        s_dev = summarize(all_rows, lambda x: x["config"] == c and x["dev_sheet"])
        s_held = summarize(all_rows, lambda x: x["config"] == c and not x["dev_sheet"])
        s_ag = summarize(all_rows, lambda x: x["config"] == c and x["stem"].startswith("GD-"))
        s_px = summarize(all_rows, lambda x: x["config"] == c and x["stem"].startswith("PX-"))
        print(f"\n{c}: all_13={s_all}  held_out_10={s_held}  dev_3={s_dev}  "
              f"AG_family={s_ag}  PX_family={s_px}")

    print("\n=== Per-sheet table ===")
    for r in sorted(all_rows, key=lambda x: (x["config"], x["stem"])):
        print(f"{r['config']:25s} {r['stem']:25s} revR={r['revR']:.3f} "
              f"sec/sheet={r['sec_per_sheet']:.0f} generations={r['n_generations']}")
else:
    print("all_rows is empty -- run §7.2 first (or the smoke cell in §7.1).")


## 9. Push results to HF, free GPU

`results.csv`/`experiments/stage4/v*.md`-style write-up happens LOCALLY on the Mac, after
Tom has reviewed these raw numbers (§10 checkpoint 3) — not from inside this notebook.


In [ ]:
from huggingface_hub import HfApi

with open("/content/extraction_agent_local_results.json", "w") as f:
    json.dump({
        "rows": all_rows,
        "reference_baselines": {
            "gpt55_high_14sheets": GPT55_HIGH_MEAN_REVR, "gpt55_low_14sheets": GPT55_LOW_MEAN_REVR,
            "sonnet46_14sheets": SONNET46_MEAN_REVR, "gemini31pro_14sheets": GEMINI31PRO_MEAN_REVR,
        },
        "dev_stems": sorted(DEV_STEMS), "smoke_stem": SMOKE_STEM,
    }, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/extraction_agent_local_results.json",
    path_in_repo="benchmarks/extraction_agent_local_results.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("results pushed to HF")

del qwen_model, qwen_processor
torch.cuda.empty_cache()

from google.colab import runtime
runtime.unassign()


In [15]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/rescore3.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

rescore3.py: 0.00B [00:00, ?B/s]


=== GD-T-435-DR-2031-030 ===  (0 tags produced by Qwen)
  sample: []
  revR=0.000  (0/63 truth tags found, 0 cleaned predictions)   [GPT-5.5-high recorded: 0.841]

=== PX-2365-0150022-001 ===  (0 tags produced by Qwen)
  sample: []
  revR=0.000  (0/60 truth tags found, 0 cleaned predictions)   [GPT-5.5-high recorded: 0.75]

=== PX-2368-0180004-001 ===  (0 tags produced by Qwen)
  sample: []
  revR=0.000  (0/50 truth tags found, 0 cleaned predictions)   [GPT-5.5-high recorded: 0.98]


In [16]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/debug_raw.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

debug_raw.py: 0.00B [00:00, ?B/s]

tags produced: 0

GENERATION 0  (prompt 16800 chars, output 62 chars)
--- prompt TAIL (last 600 chars, shows the JSON instruction) ---
le": "Text", "type": "string"}, "type": {"default": "", "title": "Type", "type": "string"}, "word_ids": {"default": [], "items": {"type": "integer"}, "title": "Word Ids", "type": "array"}, "bbox": {"default": [], "items": {"type": "number"}, "title": "Bbox", "type": "array"}}, "title": "OcrTag", "type": "object"}}, "properties": {"standard": {"default": "", "title": "Standard", "type": "string"}, "prefix": {"default": "", "title": "Prefix", "type": "string"}, "tags": {"default": [], "items": {"$ref": "#/$defs/OcrTag"}, "title": "Tags", "type": "array"}}, "title": "OcrResult", "type": "object"}
--- raw OUTPUT (first 2000 chars) ---
```json
{
  "standard": "",
  "prefix": "",
  "tags": []
}
```
GENERATION 1  (prompt 16114 chars, output 62 chars)
--- prompt TAIL (last 600 chars, shows the JSON instruction) ---
le": "Text", "type": "string"}, "type": {"defa

In [17]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/fix_and_run3.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

fix_and_run3.py: 0.00B [00:00, ?B/s]

  GD-T-435-DR-2031-030      L-ocr revR=0.810 (51/63)  tags=137  698s  4 gen   [GPT-5.5-high: 0.841]
  PX-2365-0150022-001       L-ocr revR=0.150 (9/60)  tags=149  473s  5 gen   [GPT-5.5-high: 0.75]
  PX-2368-0180004-001       L-ocr revR=0.140 (7/50)  tags=20  502s  4 gen   [GPT-5.5-high: 0.98]

MEAN over 3 sheets: L-ocr(Qwen local)=0.367  vs GPT-5.5-high(same sheets)=0.857
(reference: GPT-5.5 full-14-sheet means: high=0.836, low=0.813)


In [5]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/diagnose_px.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

diagnose_px.py: 0.00B [00:00, ?B/s]


PX-2365-0150022-001: 149 raw tags, 138 cleaned, 9/60 hit
--- PREDICTED (all 147 distinct raw texts) ---
['1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '6', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '7', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '8', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '9', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', 'MTR-2160', 'MTR-2165', 'PBE-2160', 

## Run order

1. ✅ DONE (2026-07-17): agent code pushed to `timthy45/pnid-extraction-agent-src` via
   `scripts/package_extraction_agent_src_for_colab.sh`, verified private + contents complete.
2. ✅ DONE (2026-07-17): Tom's separate explicit sign-off given on the 13 sheet PDFs
   (Restricted/EAR99); both archives pushed to `timthy45/pnid-extraction-datasets` at
   `sheets/AG_PNID.zip`/`sheets/RIVE_LTTS_Sample.zip`, verified private + present. §4's cell
   is filled in and ready.
3. **Next step:** open this notebook in Colab with a GPU runtime, paste `HF_TOKEN` into §1,
   Run All through §7.1 (smoke gate) only — `SMOKE_ONLY=True` is the default.
4. Show Tom the smoke-gate numbers. Only after an explicit go-ahead, set `SMOKE_ONLY=False`
   and re-run §7.2 onward for the full 13-sheet, 4-config run.
5. Show Tom the raw §8 numbers before writing anything to `results.csv` or
   `experiments/stage4/v*.md` (that write-up happens locally, not in this notebook).


In [12]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/fix2_and_run2.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

fix2_and_run2.py: 0.00B [00:00, ?B/s]

precomputed_ocr_words.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  PX-2365-0150022-001       L-ocr(v3) revR=0.183 (11/60)  tags=134  459s   [GPT-5.5-high: 0.75; round-1: 0.150]
  PX-2368-0180004-001       L-ocr(v3) revR=0.120 (6/50)  tags=6  465s   [GPT-5.5-high: 0.98; round-1: 0.140]

Done. 3-sheet mean = (0.810 + the two numbers above) / 3.


In [14]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/qwen_best_run2.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

qwen_best_run2.py: 0.00B [00:00, ?B/s]

precomputed_ocr_words.json: 0.00B [00:00, ?B/s]


=== PX-2365-0150022-001 (Qwen-optimized multi-pass) ===
      [pass instrument] 127 raw -> 25 new (total 25)
      [pass equipment] 18 raw -> 18 new (total 43)
      [pass line] 0 raw -> 0 new (total 43)
      [pass misc] 156 raw -> 5 new (total 48)
      [pass recovery] 162 raw -> 4 new (total 4)
      [pass instrument] 127 raw -> 1 new (total 1)
      [pass equipment] 96 raw -> 7 new (total 8)
      [pass line] 57 raw -> 1 new (total 9)
      [pass misc] 0 raw -> 0 new (total 9)
      [pass instrument] 127 raw -> 1 new (total 1)
      [pass equipment] 96 raw -> 7 new (total 8)


: 

In [1]:
import json
from pathlib import Path
_stem = "PX-2365-0150022-001"
_pp = Path(f"/content/preds_L-ocr-v4/{_stem}_p1.json")
if _pp.exists():
    _tags = json.load(open(_pp)).get("tags", [])
    _tp = Path("/content/agent_src/agents/pnid-extraction-agent/scripts/eval/review_reads") / _stem / "reviewed_truth.json"
    _pred = {a for a in (review_keep(t.get("text")) for t in _tags) if a}
    _rr, _hits, _missed = review_recall(_pred, load_reviewed_truth(str(_tp)))
    print(f"{_stem}: revR={_rr:.3f} ({_hits}/{len(load_reviewed_truth(str(_tp)))} truth) from {len(_tags)} tags [REAL, main-pass-only]")
else:
    print("preds file not written - the run died before the first sheet's pipeline write")

NameError: name 'review_keep' is not defined

In [2]:
import json, sys
from pathlib import Path
sys.path.insert(0, "/content/agent_src/agents/pnid-extraction-agent")
from scripts.eval.score import load_reviewed_truth, review_keep, review_recall

_stem = "PX-2365-0150022-001"
_tags = json.load(open(f"/content/preds_L-ocr-v4/{_stem}_p1.json")).get("tags", [])
_truth = load_reviewed_truth(f"/content/agent_src/agents/pnid-extraction-agent/scripts/eval/review_reads/{_stem}/reviewed_truth.json")
_pred = {a for a in (review_keep(t.get("text")) for t in _tags) if a}
_rr, _hits, _missed = review_recall(_pred, _truth)
print(f"{_stem}: revR={_rr:.3f} ({_hits}/{len(_truth)} truth found, {len(_tags)} tags) [REAL, tuned strategy]")

PX-2365-0150022-001: revR=0.233 (14/60 truth found, 52 tags) [REAL, tuned strategy]


In [8]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms3_run.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

arms3_run.py: 0.00B [00:00, ?B/s]

precomputed_ocr_words.json: 0.00B [00:00, ?B/s]

cached-OCR shim installed (stem-keyed)

════ ARM 1: whole-page ocr_reasoning (multi-pass) ════


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  PX-2365-0150022-001  arm1 revR=0.183 (11/60)  tags=10  2590s
  PX-2368-0180004-001  arm1 revR=0.080 (4/50)  tags=4  2593s

════ ARM 3: CV-hybrid (read_shapes + read_regions, real agent path) ════
  PX-2365-0150022-001  arm3 revR=0.467 (28/60)  tags=143  54s
  PX-2368-0180004-001  arm3 revR=0.540 (27/50)  tags=139  102s

════ ARM 2: Molmo2 512-config pointing + local crop reads ════
  loading Molmo2 (allenai/Molmo2-O-7B) - pointing on 2 sheet(s), the slow part...


...

model-00003-of-00007.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

model-00006-of-00007.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

  PX-2365-0150022-001: 389 points  1480s
  PX-2368-0180004-001: 378 points  1505s
  Molmo2 freed
  PX-2365-0150022-001  arm2 revR=0.700 (42/60)  tags=115  pts=389  252s
  PX-2368-0180004-001  arm2 revR=0.700 (35/50)  tags=91  pts=378  245s

══════════════════════════════════════════════════════════════════════════════
sheet                                arm1           arm2           arm3          UNION  GPT-5.5
PX-2365-0150022-001         0.183 (11/60)  0.700 (42/60)  0.467 (28/60)  0.850 (51/60)  0.75
PX-2368-0180004-001          0.080 (4/50)  0.700 (35/50)  0.540 (27/50)  0.760 (38/50)  0.98

raw per-arm texts saved to /content/arms3_texts.json - paste this table back.


In [1]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/diagnose_union.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

NameError: name 'HF_TOKEN' is not defined

In [8]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms4_run.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

arms4_run.py: 0.00B [00:00, ?B/s]

precomputed_ocr_words.json: 0.00B [00:00, ?B/s]

cached-OCR shim installed (stem-keyed)

════ ARM 1 (cheap single-pass): whole-page ocr_reasoning ════


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  PX-2365-0150022-001  arm1 revR=0.150 (9/60)  tags=148  537s
  PX-2368-0180004-001  arm1 revR=0.120 (6/50)  tags=12  557s

════ ARM 0: deterministic regex assembly (no model) ════
  PX-2365-0150022-001  arm0 revR=0.167 (10/60)  cands=505  0.0s
  PX-2368-0180004-001  arm0 revR=0.200 (10/50)  cands=90  0.0s

════ ARM 3: CV-hybrid (read_shapes + read_regions, real agent path) ════
  PX-2365-0150022-001  arm3 revR=0.467 (28/60)  tags=143  65s
  PX-2368-0180004-001  arm3 revR=0.540 (27/50)  tags=139  120s

════ ARM 2: Molmo2 512-config pointing + local crop reads ════
  PX-2365-0150022-001: cached points loaded (389 pts)
  PX-2368-0180004-001: cached points loaded (378 pts)
  masked ROUND-2 pointing on 2 sheet(s) - loading Molmo2 again...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 344.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 68.00 MiB is free. Process 3764 has 48.90 GiB memory in use. Including non-PyTorch memory, this process has 30.27 GiB memory in use. Of the allocated memory 29.36 GiB is allocated by PyTorch, and 416.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [9]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms4_resume.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

arms4_resume.py: 0.00B [00:00, ?B/s]

[resume] freeing cached CUDA memory + moving Qwen off-GPU before Molmo2 round-2 load
  after clear: reserved=14.4GB allocated=14.0GB
  masked ROUND-2 pointing on 2 sheet(s) - loading Molmo2 again...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 68.00 MiB is free. Process 3764 has 48.90 GiB memory in use. Including non-PyTorch memory, this process has 30.27 GiB memory in use. Of the allocated memory 29.38 GiB is allocated by PyTorch, and 395.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [1]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms4_finalize_r1only.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

NameError: name 'HF_TOKEN' is not defined

In [1]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id="timthy45/pnid-extraction-agent-src", filename="colab_cells/arms_final.py", repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

NameError: name 'HF_TOKEN' is not defined